In [10]:
import cv2
import numpy as np
from ai_edge_litert.interpreter import Interpreter, load_delegate

DET_PATH = 'models/mediapipe_hand_gesture-palm_detector-w8a8.tflite'
DET_COORD_SCALE = 1.590442;  DET_COORD_ZP = 51
DET_SCORE_SCALE = 0.003906;  DET_SCORE_ZP = 0

def generate_anchors():
    anchors = []
    for grid, n in [(32, 2), (16, 2), (8, 6)]:
        for y in range(grid):
            for x in range(grid):
                cx, cy = (x + 0.5) / grid, (y + 0.5) / grid
                for _ in range(n):
                    anchors.append([cx, cy])
    return np.array(anchors, dtype=np.float32)

ANCHORS = generate_anchors()

interp = Interpreter(model_path=DET_PATH)
interp.allocate_tensors()
det_in  = interp.get_input_details()
det_out = {t['name']: t['index'] for t in interp.get_output_details()}

cap = cv2.VideoCapture(2)
ret, frame = cap.read()
cap.release()

if not ret:
    print('Camera failed')
    exit()

print(f'Frame shape: {frame.shape}')

inp = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (256, 256))[np.newaxis]
interp.set_tensor(det_in[0]['index'], inp)
interp.invoke()

scores = (interp.get_tensor(det_out['box_scores'])[0].astype(np.float32) - DET_SCORE_ZP) * DET_SCORE_SCALE
best = int(np.argmax(scores))
print(f'Best score: {scores[best]:.4f}')
print(f'Top 5 scores: {sorted(scores, reverse=True)[:5]}')

coords = (interp.get_tensor(det_out['box_coords'])[0].astype(np.float32) - DET_COORD_ZP) * DET_COORD_SCALE
cx = coords[best, 0] / 256.0 + ANCHORS[best, 0]
cy = coords[best, 1] / 256.0 + ANCHORS[best, 1]
print(f'Best box: cx={cx:.3f} cy={cy:.3f}')
print(f'Passes 0.7 threshold: {scores[best] > 0.7}')
print(f'Passes 0.3 threshold: {scores[best] > 0.3}')

Frame shape: (480, 640, 3)
Best score: 0.9960
Top 5 scores: [np.float32(0.99603), np.float32(0.99603), np.float32(0.99603), np.float32(0.99603), np.float32(0.99603)]
Best box: cx=0.549 cy=1.246
Passes 0.7 threshold: True
Passes 0.3 threshold: True


In [11]:

import cv2, numpy as np
from ai_edge_litert.interpreter import Interpreter

DET_PATH = 'models/mediapipe_hand_gesture-palm_detector-w8a8.tflite'
LM_PATH  = 'models/mediapipe_hand_gesture-hand_landmark_detector-w8a8.tflite'
DET_COORD_SCALE = 1.590442; DET_COORD_ZP = 51
DET_SCORE_SCALE = 0.003906; DET_SCORE_ZP = 0
LM_LM_SCALE = 0.972075;    LM_LM_ZP    = 33
LM_SCORE_SCALE = 0.003906; LM_SCORE_ZP = 0

def generate_anchors():
    anchors = []
    for grid, n in [(32, 2), (16, 2), (8, 6)]:
        for y in range(grid):
            for x in range(grid):
                cx, cy = (x + 0.5) / grid, (y + 0.5) / grid
                for _ in range(n):
                    anchors.append([cx, cy])
    return np.array(anchors, dtype=np.float32)

ANCHORS = generate_anchors()

det = Interpreter(model_path=DET_PATH); det.allocate_tensors()
lm  = Interpreter(model_path=LM_PATH);  lm.allocate_tensors()
det_in  = det.get_input_details()
det_out = {t['name']: t['index'] for t in det.get_output_details()}
lm_in   = lm.get_input_details()
lm_out  = {t['name']: t['index'] for t in lm.get_output_details()}

cap = cv2.VideoCapture(2)
ret, frame = cap.read()
cap.release()
fh, fw = frame.shape[:2]
print(f'Frame: {fw}x{fh}')

inp = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (256,256))[np.newaxis]
det.set_tensor(det_in[0]['index'], inp)
det.invoke()

scores = (det.get_tensor(det_out['box_scores'])[0].astype(np.float32) - DET_SCORE_ZP) * DET_SCORE_SCALE
best   = int(np.argmax(scores))
coords = (det.get_tensor(det_out['box_coords'])[0].astype(np.float32) - DET_COORD_ZP) * DET_COORD_SCALE
print(f'Score: {scores[best]:.4f}')
print(f'Raw coord offset (pixels in 256 space): cx={coords[best,0]:.1f} cy={coords[best,1]:.1f} w={coords[best,2]:.1f} h={coords[best,3]:.1f}')

# Try different decoding approaches
cx_a = coords[best, 0] / 256.0 + ANCHORS[best, 0]
cy_a = coords[best, 1] / 256.0 + ANCHORS[best, 1]
print(f'Decoded (current): cx={cx_a:.3f} cy={cy_a:.3f}')

# Try: maybe coords are already normalized (not pixel space)
cx_b = coords[best, 0] + ANCHORS[best, 0]
cy_b = coords[best, 1] + ANCHORS[best, 1]
print(f'Decoded (no /256): cx={cx_b:.3f} cy={cy_b:.3f}')

# Try: maybe x/y are swapped
cx_c = coords[best, 1] / 256.0 + ANCHORS[best, 1]
cy_c = coords[best, 0] / 256.0 + ANCHORS[best, 0]
print(f'Decoded (xy swap): cx={cx_c:.3f} cy={cy_c:.3f}')

# Show anchor of best detection
print(f'Anchor: {ANCHORS[best]}')
print(f'Anchor index: {best}')

# Now try cropping with the current decode and see what happens
side = max(coords[best,2], coords[best,3]) / 256.0 * 2.2
print(f'Crop side (normalized): {side:.3f}')
x1 = max(0, int((cx_a - side/2) * fw))
y1 = max(0, int((cy_a - side/2) * fh))
x2 = min(fw, int((cx_a + side/2) * fw))
y2 = min(fh, int((cy_a + side/2) * fh))
print(f'Crop pixels: ({x1},{y1}) -> ({x2},{y2}), size={x2-x1}x{y2-y1}')
crop = frame[y1:y2, x1:x2]
print(f'Crop array size: {crop.size}')

if crop.size > 0:
    lm_inp = cv2.resize(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB), (224,224))[np.newaxis]
    lm.set_tensor(lm_in[0]['index'], lm_inp)
    lm.invoke()
    lm_score = float(lm.get_tensor(lm_out['scores'])[0,0] - LM_SCORE_ZP) * LM_SCORE_SCALE
    print(f'Landmark score: {lm_score:.4f}')
    lm_vals = ((lm.get_tensor(lm_out['landmarks'])[0].astype(np.float32) - LM_LM_ZP) * LM_LM_SCALE).reshape(21,3)
    print(f'Wrist: {lm_vals[0]}')
    print(f'Index tip: {lm_vals[8]}')
else:
    print('Crop is empty — this is why nothing draws')


Frame: 640x480
Score: 0.9960
Raw coord offset (pixels in 256 space): cx=46.1 cy=151.1 w=108.2 h=213.1
Decoded (current): cx=0.461 cy=1.246
Decoded (no /256): cx=46.404 cy=151.748
Decoded (xy swap): cx=1.246 cy=0.461
Anchor: [0.28125 0.65625]
Anchor index: 2376
Crop side (normalized): 1.831
Crop pixels: (0,158) -> (640,480), size=640x322
Crop array size: 618240
Landmark score: 0.9804
Wrist: [ 63.184875 182.75009    0.      ]
Index tip: [ 94.291275  25.273949 -19.4415  ]


In [12]:
import cv2, numpy as np
from ai_edge_litert.interpreter import Interpreter

LM_PATH = 'models/mediapipe_hand_gesture-hand_landmark_detector-w8a8.tflite'
LM_LM_SCALE = 0.972075; LM_LM_ZP = 33
LM_SCORE_SCALE = 0.003906; LM_SCORE_ZP = 0

interp = Interpreter(model_path=LM_PATH)
interp.allocate_tensors()
lm_in  = interp.get_input_details()
lm_out = {t['name']: t['index'] for t in interp.get_output_details()}

cap = cv2.VideoCapture(2)
ret, frame = cap.read()
cap.release()

inp = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (224,224))[np.newaxis]
interp.set_tensor(lm_in[0]['index'], inp)
interp.invoke()

score = float(interp.get_tensor(lm_out['scores'])[0,0] - LM_SCORE_ZP) * LM_SCORE_SCALE
lm = ((interp.get_tensor(lm_out['landmarks'])[0].astype(np.float32) - LM_LM_ZP) * LM_LM_SCALE).reshape(21,3)

print(f'Score: {score:.4f}')
print(f'Raw landmark range: x=[{lm[:,0].min():.2f},{lm[:,0].max():.2f}] y=[{lm[:,1].min():.2f},{lm[:,1].max():.2f}]')
print('All landmarks:')
for i, l in enumerate(lm):
    print(f'  {i:2d}: x={l[0]:.3f} y={l[1]:.3f} z={l[2]:.3f}')


Score: 0.9804
Raw landmark range: x=[34.99,126.37] y=[74.85,202.19]
All landmarks:
   0: x=59.297 y=202.192 z=0.000
   1: x=81.654 y=194.415 z=-4.860
   2: x=101.096 y=182.750 z=-6.805
   3: x=115.677 y=177.890 z=-10.693
   4: x=126.370 y=178.862 z=-13.609
   5: x=87.487 y=137.063 z=-3.888
   6: x=93.319 y=113.733 z=-5.832
   7: x=95.263 y=102.068 z=-7.777
   8: x=97.207 y=88.459 z=-10.693
   9: x=74.850 y=133.174 z=-2.916
  10: x=72.906 y=103.040 z=-4.860
  11: x=71.934 y=87.487 z=-7.777
  12: x=72.906 y=74.850 z=-9.721
  13: x=62.213 y=135.118 z=-3.888
  14: x=57.352 y=109.844 z=-5.832
  15: x=55.408 y=95.263 z=-7.777
  16: x=54.436 y=81.654 z=-8.749
  17: x=49.576 y=142.895 z=-4.860
  18: x=42.771 y=124.426 z=-6.805
  19: x=38.883 y=114.705 z=-7.777
  20: x=34.995 y=103.040 z=-8.749


In [18]:
from ai_edge_litert.interpreter import Interpreter
for name, path in [
    ('detector', 'models/old/HandDetector.tflite'),
    ('landmark', 'models/old/HandLandmarkDetector.tflite'),
]:
    print(f'\n=== {name} ===')
    interp = Interpreter(model_path=path)
    interp.allocate_tensors()
    print('INPUTS:')
    
    for t in interp.get_input_details():
        print(f'  {t["name"]}: shape={t["shape"]}, dtype={t["dtype"]}')
    print('OUTPUTS:')
    for t in interp.get_output_details():
        print(f'  {t["name"]}: shape={t["shape"]}, dtype={t["dtype"]}')



=== detector ===
INPUTS:
  image: shape=[  1 256 256   3], dtype=<class 'numpy.float32'>
OUTPUTS:
  box_coords: shape=[   1 2944   18], dtype=<class 'numpy.float32'>
  box_scores: shape=[   1 2944    1], dtype=<class 'numpy.float32'>

=== landmark ===
INPUTS:
  image: shape=[  1 256 256   3], dtype=<class 'numpy.float32'>
OUTPUTS:
  scores: shape=[1], dtype=<class 'numpy.float32'>
  lr: shape=[1], dtype=<class 'numpy.float32'>
  landmarks: shape=[ 1 21  3], dtype=<class 'numpy.float32'>
